In [2]:
import tkinter as tk
from tkinter import messagebox
import heapq
import math
import time
import random

CELL = 32

def manhattan(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def euclidean(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

def get_neighbors(grid, r, c):
    result = []
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r+dr, c+dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]):
            if grid[nr][nc] != 1:
                result.append((nr, nc))
    return result

def greedy(grid, start, goal, h):
    open_list = [(h(start, goal), start)]
    came_from = {start: None}
    visited = []
    closed = set()
    while open_list:
        f, current = heapq.heappop(open_list)
        if current in closed:
            continue
        closed.add(current)
        visited.append(current)
        if current == goal:
            path = []
            node = goal
            while node:
                path.append(node)
                node = came_from[node]
            return list(reversed(path)), visited
        for nb in get_neighbors(grid, *current):
            if nb not in came_from:
                came_from[nb] = current
                heapq.heappush(open_list, (h(nb, goal), nb))
    return None, visited

def astar(grid, start, goal, h):
    open_list = [(h(start, goal), 0, start)]
    came_from = {start: None}
    g_score = {start: 0}
    visited = []
    closed = set()
    while open_list:
        f, g, current = heapq.heappop(open_list)
        if current in closed:
            continue
        closed.add(current)
        visited.append(current)
        if current == goal:
            path = []
            node = goal
            while node:
                path.append(node)
                node = came_from[node]
            return list(reversed(path)), visited
        for nb in get_neighbors(grid, *current):
            new_g = g + 1
            if nb not in g_score or new_g < g_score[nb]:
                g_score[nb] = new_g
                came_from[nb] = current
                heapq.heappush(open_list, (new_g + h(nb, goal), new_g, nb))
    return None, visited


class App:
    def __init__(self, root):
        self.root = root
        self.root.title("Pathfinding - Iteration 2")
        self.rows = 15
        self.cols = 20
        self.start = (1, 1)
        self.goal = (13, 18)
        self.grid = [[0]*self.cols for _ in range(self.rows)]
        self.grid[self.start[0]][self.start[1]] = 2
        self.grid[self.goal[0]][self.goal[1]] = 3
        self.algorithm = tk.StringVar(value="A*")
        self.heuristic = tk.StringVar(value="Manhattan")
        self.edit_mode = tk.StringVar(value="wall")
        self.density = tk.DoubleVar(value=0.3)
        self.after_id = None
        self.build_ui()
        self.draw_grid()

    def build_ui(self):
        left = tk.Frame(self.root, padx=10, pady=10)
        left.pack(side=tk.LEFT, fill=tk.Y)

        tk.Label(left, text="Algorithm").pack(anchor="w")
        for a in ["A*", "Greedy"]:
            tk.Radiobutton(left, text=a, variable=self.algorithm, value=a).pack(anchor="w")

        tk.Label(left, text="Heuristic").pack(anchor="w", pady=(8,0))
        for h in ["Manhattan", "Euclidean"]:
            tk.Radiobutton(left, text=h, variable=self.heuristic, value=h).pack(anchor="w")

        tk.Label(left, text="Edit Mode").pack(anchor="w", pady=(8,0))
        for m, l in [("wall","Wall"), ("start","Move Start"), ("goal","Move Goal")]:
            tk.Radiobutton(left, text=l, variable=self.edit_mode, value=m).pack(anchor="w")

        tk.Label(left, text="Obstacle Density").pack(anchor="w", pady=(8,0))
        tk.Scale(left, variable=self.density, from_=0.0, to=0.6, resolution=0.05, orient=tk.HORIZONTAL).pack(anchor="w")

        tk.Button(left, text="Generate Maze", command=self.generate_maze, width=14).pack(pady=3)
        tk.Button(left, text="Clear Grid", command=self.clear_grid, width=14).pack(pady=3)
        tk.Button(left, text="Run", command=self.run, width=14).pack(pady=3)
        tk.Button(left, text="Stop", command=self.stop, width=14).pack(pady=3)

        tk.Label(left, text="").pack()
        self.lbl_nodes = tk.Label(left, text="Nodes visited: -")
        self.lbl_nodes.pack(anchor="w")
        self.lbl_cost = tk.Label(left, text="Path cost: -")
        self.lbl_cost.pack(anchor="w")
        self.lbl_time = tk.Label(left, text="Time: -")
        self.lbl_time.pack(anchor="w")

        tk.Label(left, text="").pack()
        tk.Label(left, text="Green = Start").pack(anchor="w")
        tk.Label(left, text="Red = Goal").pack(anchor="w")
        tk.Label(left, text="Blue = Visited").pack(anchor="w")
        tk.Label(left, text="Yellow = Frontier").pack(anchor="w")
        tk.Label(left, text="Cyan = Path").pack(anchor="w")

        self.canvas = tk.Canvas(self.root, width=self.cols*CELL, height=self.rows*CELL, bg="white")
        self.canvas.pack(side=tk.LEFT, padx=10, pady=10)
        self.canvas.bind("<Button-1>", self.on_click)
        self.canvas.bind("<B1-Motion>", self.on_drag)

    def draw_grid(self):
        self.canvas.delete("all")
        colors = {0:"white", 1:"#444444", 2:"green", 3:"red", 4:"#aac4ff", 5:"yellow", 6:"cyan"}
        for r in range(self.rows):
            for c in range(self.cols):
                x1, y1 = c*CELL, r*CELL
                self.canvas.create_rectangle(x1, y1, x1+CELL, y1+CELL,
                    fill=colors.get(self.grid[r][c], "white"), outline="gray")
        mid = CELL // 2
        self.canvas.create_text(self.start[1]*CELL+mid, self.start[0]*CELL+mid, text="S", font=("Arial",10,"bold"))
        self.canvas.create_text(self.goal[1]*CELL+mid, self.goal[0]*CELL+mid, text="G", font=("Arial",10,"bold"))

    def cell_from_event(self, event):
        r, c = event.y // CELL, event.x // CELL
        if 0 <= r < self.rows and 0 <= c < self.cols:
            return r, c
        return None

    def on_click(self, event):
        pos = self.cell_from_event(event)
        if not pos:
            return
        r, c = pos
        mode = self.edit_mode.get()
        if mode == "wall" and (r, c) not in (self.start, self.goal):
            self.grid[r][c] = 0 if self.grid[r][c] == 1 else 1
            self.draw_grid()
        elif mode == "start":
            self.grid[self.start[0]][self.start[1]] = 0
            self.start = (r, c)
            self.grid[r][c] = 2
            self.draw_grid()
        elif mode == "goal":
            self.grid[self.goal[0]][self.goal[1]] = 0
            self.goal = (r, c)
            self.grid[r][c] = 3
            self.draw_grid()

    def on_drag(self, event):
        pos = self.cell_from_event(event)
        if pos and self.edit_mode.get() == "wall":
            r, c = pos
            if (r, c) not in (self.start, self.goal):
                self.grid[r][c] = 1
                self.draw_grid()

    def clear_grid(self):
        self.stop()
        self.grid = [[0]*self.cols for _ in range(self.rows)]
        self.grid[self.start[0]][self.start[1]] = 2
        self.grid[self.goal[0]][self.goal[1]] = 3
        self.draw_grid()

    def generate_maze(self):
        self.stop()
        for r in range(self.rows):
            for c in range(self.cols):
                if (r, c) not in (self.start, self.goal):
                    self.grid[r][c] = 1 if random.random() < self.density.get() else 0
        self.grid[self.start[0]][self.start[1]] = 2
        self.grid[self.goal[0]][self.goal[1]] = 3
        self.draw_grid()

    def reset_colors(self):
        for r in range(self.rows):
            for c in range(self.cols):
                if self.grid[r][c] in (4, 5, 6):
                    self.grid[r][c] = 0
        self.grid[self.start[0]][self.start[1]] = 2
        self.grid[self.goal[0]][self.goal[1]] = 3

    def stop(self):
        if self.after_id:
            self.root.after_cancel(self.after_id)
            self.after_id = None

    def run(self):
        self.stop()
        self.reset_colors()
        h = manhattan if self.heuristic.get() == "Manhattan" else euclidean
        t0 = time.time()
        if self.algorithm.get() == "A*":
            path, visited = astar(self.grid, self.start, self.goal, h)
        else:
            path, visited = greedy(self.grid, self.start, self.goal, h)
        elapsed = (time.time() - t0) * 1000
        self.lbl_nodes.config(text=f"Nodes visited: {len(visited)}")
        self.lbl_cost.config(text=f"Path cost: {len(path)-1 if path else 'No path'}")
        self.lbl_time.config(text=f"Time: {elapsed:.2f} ms")
        if not path:
            messagebox.showwarning("No Path", "Cannot reach the goal!")
            return
        self.animate_visited(visited, path, 0)

    def animate_visited(self, visited, path, i):
        if i < len(visited):
            r, c = visited[i]
            if (r, c) not in (self.start, self.goal):
                self.grid[r][c] = 4
            self.draw_grid()
            self.after_id = self.root.after(20, self.animate_visited, visited, path, i+1)
        else:
            self.animate_path(path, 0)

    def animate_path(self, path, i):
        if i < len(path):
            r, c = path[i]
            if (r, c) not in (self.start, self.goal):
                self.grid[r][c] = 6
            self.draw_grid()
            self.after_id = self.root.after(40, self.animate_path, path, i+1)


root = tk.Tk()
app = App(root)
root.mainloop()